### Install requiremnts

In [23]:
!pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [24]:
import sys
sys.path.append('../')
from checkpoints import DT, MT, LT
import os

from validate_birdset import load_model, get_test_loader, test
from checkpoints import DT, MT, LT
import numpy as np
import subprocess


DOWN_TASKS = ["HSN", "POW", "SNE", "PER", "NES", "UHH", "NBP", "SSW"]

#### Download checkpoints

In [25]:

all_checkpoint_available = (
    all(os.path.exists( '../' + p) for ds in DT for p in DT[ds]) and
    all(os.path.exists('../' + p) for p in MT['ALL']) and
    all(os.path.exists('../' + p) for p in LT['ALL'])
)
if not all_checkpoint_available:
   subprocess.run(["python", "prepare_checkpoints.py"], cwd="..")
else:
   print("All checkpoints available")

All checkpoints available


In [26]:
def run_testing(regime, device='cuda'):
    for down_task in DOWN_TASKS:

        # Get checkpoint paths for this task (empty list if task not present)
        ckpts = regime.get(down_task, [])
        print(ckpts)
        if not ckpts:
            ckpts  = regime['ALL']

        models = []
        configs = []

        # ---------------------------------------------------------
        # Load models and their configs from checkpoints
        # ---------------------------------------------------------
        for ckpt in ckpts:
            m, cfg = load_model('../' + ckpt, device=device)
            models.append(m)
            # Override validation parameters for evaluation
            cfg.frontend.val_target_length = 701
            cfg.event_decoder.val.extracted_interval = 7
            # Set dataset name to current downstream task
            cfg.train.dataset_name = down_task
            configs.append(cfg)

        results = dict()
        # Create test dataloader using the first config
        val_loader, class_names = get_test_loader(configs[0])

        # Map class names to label IDs using the config label map
        label2id = {k: v for k, v in configs[0].train.label_map.items()}

        # Only test on classes relevant for the current dataset
        relevant = [label2id[c] for c in class_names]

        # Store metrics for each model (for later averaging)
        metrics = {"auroc": [],
                   "cmap": [],
                   "top1_acc": []}

        # ---------------------------------------------------------
        # test each model on the test dataset
        # ---------------------------------------------------------
        for model, _ in zip(models, configs):
            # Run evaluation
            auroc, cmap, top1_acc = test(model, val_loader, relevant, device=device)

            # Store metrics
            metrics["auroc"].append(auroc)
            metrics["cmap"].append(cmap)
            metrics["top1_acc"].append(top1_acc)

        # ---------------------------------------------------------
        # Aggregate metrics across all checkpoints/models
        # ---------------------------------------------------------
        results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
        print(results)

In [27]:
# DT REGIME
# for cpu: run_testing(DT, device='cpu')
run_testing(DT)

['checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', 'checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', 'checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-03-14 15:03:58,064 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:03:58,242 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 15:03:58,257 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 15:03:58,660 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:03:58,776 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 15:03:58,791 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 15:03:59,211 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:03:59,326 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}
['checkpoints/DT/POW/POW_eca_nfnet_l1_2025-10-11_143357', 'checkpoints/DT/POW/POW_eca_nfnet_l1_2025-10-11_145846', 'checkpoints/DT/POW/POW_eca_nfnet_l1_2025-10-11_152335']


2026-03-14 15:06:18,566 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:06:18,680 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 15:06:18,695 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 15:06:19,095 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:06:19,210 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 15:06:19,223 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 15:06:19,627 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:06:19,747 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'POW': {'auroc': 0.9412414127927123, 'cmap': 0.6017426453929788, 'top1_acc': 0.9455520311991373}}
['checkpoints/DT/SNE/SNE_eca_nfnet_l1_2025-10-12_131606', 'checkpoints/DT/SNE/SNE_eca_nfnet_l1_2025-10-12_135017', 'checkpoints/DT/SNE/SNE_eca_nfnet_l1_2025-10-12_142429']


...

{'UHH': {'auroc': 0.8808524887362056, 'cmap': 0.34541931200778314, 'top1_acc': 0.6783677339553833}}
['checkpoints/DT/NBP/NBP_eca_nfnet_l1_2025-10-13_143637', 'checkpoints/DT/NBP/NBP_eca_nfnet_l1_2025-10-13_151856', 'checkpoints/DT/NBP/NBP_eca_nfnet_l1_2025-10-13_160131']


2026-03-14 15:25:26,235 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:25:26,405 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 15:25:26,419 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 15:25:26,810 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:25:26,930 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 15:25:26,944 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 15:25:27,337 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:25:27,456 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'NBP': {'auroc': 0.9270734881955889, 'cmap': 0.7104421590871208, 'top1_acc': 0.7031539877255758}}
['checkpoints/DT/SSW/SSW_eca_nfnet_l1_2025-10-11_215626', 'checkpoints/DT/SSW/SSW_eca_nfnet_l1_2025-10-11_203307', 'checkpoints/DT/SSW/SSW_eca_nfnet_l1_2025-10-11_190939']


2026-03-14 15:25:42,719 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:25:42,855 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 15:25:42,869 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 15:25:43,309 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:25:43,450 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 15:25:43,463 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 15:25:43,904 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:25:44,049 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'SSW': {'auroc': 0.9760789338403836, 'cmap': 0.5169989847925569, 'top1_acc': 0.782464603583018}}


In [28]:
run_testing(MT)

[]


2026-03-14 15:58:51,945 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:58:52,135 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 15:58:52,150 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 15:58:52,685 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:58:52,798 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 15:58:52,812 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 15:58:53,215 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 15:58:53,334 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9155463593918735, 'cmap': 0.555441695709684, 'top1_acc': 0.7048540115356445}}
[]


2026-03-14 16:01:14,560 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:01:14,680 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 16:01:14,694 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 16:01:15,112 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:01:15,230 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 16:01:15,246 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 16:01:15,658 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:01:15,776 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'POW': {'auroc': 0.9373619546116426, 'cmap': 0.539876956937427, 'top1_acc': 0.9148920178413391}}
[]


...

{'UHH': {'auroc': 0.879432883644678, 'cmap': 0.3408232896082221, 'top1_acc': 0.7133174737294515}}
[]


2026-03-14 16:20:33,565 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:20:33,775 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 16:20:33,789 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 16:20:34,203 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:20:34,321 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 16:20:34,335 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 16:20:34,874 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:20:34,989 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'NBP': {'auroc': 0.9356312438001174, 'cmap': 0.6891143922453947, 'top1_acc': 0.7068645556767782}}
[]


2026-03-14 16:20:50,431 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:20:50,542 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 16:20:50,547 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 16:20:50,960 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:20:51,078 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 16:20:51,093 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 16:20:51,497 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:20:51,608 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'SSW': {'auroc': 0.9742244681407527, 'cmap': 0.48227713453227805, 'top1_acc': 0.7558941046396891}}


In [29]:
run_testing(LT)

[]


2026-03-14 16:54:36,910 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:54:37,075 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 16:54:37,086 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 16:54:37,965 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:54:38,085 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 16:54:38,099 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 16:54:38,959 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:54:39,076 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9351716319001824, 'cmap': 0.544447258014205, 'top1_acc': 0.7214059829711914}}
[]


2026-03-14 16:57:34,154 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:57:34,274 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 16:57:34,286 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 16:57:35,017 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:57:35,130 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 16:57:35,145 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 16:57:36,011 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 16:57:36,129 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'POW': {'auroc': 0.9060017833727113, 'cmap': 0.5435451443436529, 'top1_acc': 0.9434375564257304}}
[]


...

{'UHH': {'auroc': 0.8961643650612414, 'cmap': 0.32242848432383525, 'top1_acc': 0.6951840321222941}}
[]


2026-03-14 17:21:50,864 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 17:21:51,075 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 17:21:51,094 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 17:21:51,830 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 17:21:51,950 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 17:21:51,965 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 17:21:52,862 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 17:21:52,985 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'NBP': {'auroc': 0.9363050674754332, 'cmap': 0.7039434473669259, 'top1_acc': 0.7223252852757772}}
[]


2026-03-14 17:22:11,565 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 17:22:11,694 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 17:22:11,698 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 17:22:12,435 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 17:22:12,554 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 17:22:12,570 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 17:22:13,462 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 17:22:13,582 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'SSW': {'auroc': 0.9707822561636936, 'cmap': 0.4497012047816658, 'top1_acc': 0.760113259156545}}
